In [ ]:
import sys
!{sys.executable} -m pip install pandas

In [ ]:
import os
import ast
import pandas as pd
import json
import numpy as np
from neo4j import GraphDatabase
import regex

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

In [ ]:
uri = "bolt://neo4j-gds-apoc-n10s:7687"
username = "neo4j"
password = "neo4jpassword"

In [ ]:
driver = GraphDatabase.driver(uri, auth=(username, password))

In [ ]:
DATA_DIR = "/app/notebooks/rdb"

### neo4j 초기화
- 아래 코드를 통해 넣을 수 없는 데이터(다른 방법으로 이미 넣어둔 데이터)가 있는 경우 아래 코드는 실행하면 안 됨

In [ ]:
with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print(result.single())

In [ ]:
with driver.session() as session:
    # 모든 관계와 노드 제거
    result = session.run("MATCH (n) DETACH DELETE n")
    print(result.single())

### 데이터 로딩
- 스키마 참고
  - https://confluence.tde.sktelecom.com/pages/viewpage.action?pageId=734203873
- 순서
  - (1) PRODUCT + PRICE
  - (2) VOICE, SMS, DATA, TOPUP, CUSTOMERCONDITION, PRODUCT_GROUP, BENEFITCONDITION, DEDUCTIBLE, RELATION

### PRODUCT + PRICE
- 모바일 요금제 상품
- Label: 요금제

In [ ]:
product_table = pd.read_csv(os.path.join(DATA_DIR, "PRODUCT.csv"))
product_table.head(1)

In [ ]:
product_table[product_table["productName"].str.contains("0틴플랜")]

In [ ]:
price_table = pd.read_csv(os.path.join(DATA_DIR, "PRICE.csv"))
price_table.head(1)

In [ ]:
product_table = product_table.merge(price_table, on="pmProductID", how="left")

In [ ]:
eng_colnames = [
    'pmProductID', 'mappedProductCode', 'generation', 'marketingKeyword', 
    'productName', 'productNameInEnglish', 'lineup', 'classifiedGroup',
    'productDescription', 'productSubscriptionCondition', 'statusOfOperation',
    'monthlyPrice', 'monthlyPriceWithoutVAT', 'monthlyPriceWithSelectableInstallment', 
    'billingMethod', 'netPrice'
]

kor_colnames = [
    '고유ID', '상품코드매핑', '통신규격', '마케팅키워드',
    '상품명', '영문상품명', '라인업', '상품분류',
    '상품설명', '상품가입조건', '운영상태',
    '월정액', '부가세제외월정액', '선택약정할인포함부가세제외월정액', 
    '청구방법', 'net가격'
]

kor_cols_map = {x:y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
def type_cast(input_data):
    new_data = None
    if pd.isna(input_data):
        input_data = "[]"
    elif '[' in input_data and ']' in input_data and 'nan' in input_data:
        input_data =  input_data.replace("nan", "")
    
    new_data = ast.literal_eval(input_data)

    return new_data

In [ ]:
# 리스트형으로 변환
product_table["generation"] = product_table["generation"].apply(lambda x: type_cast(x))
product_table["marketingKeyword"] = product_table["marketingKeyword"].apply(lambda x: type_cast(x))
product_table["mappedProductCode"] = product_table["mappedProductCode"].apply(lambda x: type_cast(x))

In [ ]:
with driver.session() as session:
    for _, row in product_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}
        lineup = properties.pop("라인업", None)
        product_id = properties.get("고유ID")

        if product_id:
            if lineup:
                session.run(
                    """
                    MERGE (p:요금제 {고유ID: $product_id})
                    SET p += $props
                    MERGE (l:라인업 {라인업명: $lineup})
                    MERGE (p)-[:속함]->(l)
                    """,
                    product_id=product_id,
                    props=properties,
                    lineup=lineup
                )
            else:
                session.run(
                    """
                    MERGE (p:요금제 {고유ID: $product_id})
                    SET p += $props
                    """,
                    product_id=product_id,
                    props=properties
                )

### autoProductChange

In [ ]:
PROD_INFO_PATH = "notebooks/dataload/prod_info_0219.json"

In [ ]:
with open(PROD_INFO_PATH, "rt") as fIn:
    prod_info = json.load(fIn)

In [ ]:
autoProductChange_list = []
for _prod in prod_info:
    if _prod.get("autoProductChange", {}).get("changeRule", {}).get("dateBase", {}):
        pmProductID = _prod["pmProductID"]
        dateBase = _prod["autoProductChange"]["changeRule"]["dateBase"]["value"]
        changeDate = _prod["autoProductChange"]["changeRule"]["date"]["value"]
        pmProductIDAfterChange = _prod["autoProductChange"]["productNameAfterChange"]["pmProductId"]
        legacyProductIdAfterChange = _prod["autoProductChange"]["productNameAfterChange"]["legacyProductId"]
        productNameAfterChange = _prod["autoProductChange"]["productNameAfterChange"]["productName"]
        autoProductChange_list.append({
            "pmProductID": pmProductID, 
            "dateBase": dateBase,
            "changeDate": changeDate,
            "pmProductIDAfterChange": pmProductIDAfterChange,
            # "legacyProductIdAfterChange": legacyProductIdAfterChange,
            # "productNameAfterChange": productNameAfterChange,
        })

In [ ]:
autoProductChange_table = pd.DataFrame(autoProductChange_list)
autoProductChange_table.head(10)

In [ ]:
eng_colnames = ['pmProductID', 'dateBase', 'changeDate', 
                'pmProductIDAfterChange']

kor_colnames = ['고유ID', '기준일', '변경일', 
                '변경후고유ID']

kor_cols_map = {x:y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
with driver.session() as session:
    for _, row in autoProductChange_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}
        session.run(
            """
            MATCH (pa:`요금제` {`고유ID`: $pmProductID})
            MATCH (pb:`요금제` {`고유ID`: $pmProductIDAfterChange})
            MERGE (pa)-[r:`상품자동변경`]->(pb)
            SET r.`기준일` = $dateBase,
                r.`변경일` = $changeDate
            """,
            pmProductID = properties['고유ID'],
            pmProductIDAfterChange = properties['변경후고유ID'],
            dateBase = properties['기준일'],
            changeDate = properties['변경일'],
        )

### PRICE
- 상품 가격 정보
- Label: 요금
- 관계
  - 요금제 -[:요금정보]-> 요금

In [ ]:
# price_table = pd.read_csv(os.path.join(DATA_DIR, "PRICE.csv"))
# price_table.head(1)

In [ ]:
# eng_colnames = ['monthlyPrice', 'monthlyPriceWithoutVAT', 'monthlyPriceWithSelectableInstallment', 
#                 'billingMethod', 'netPrice']

# kor_colnames = ['월정액', '부가세제외월정액', '선택약정할인포함부가세제외월정액', 
#                 '청구방법', 'net가격']

# kor_cols_map = {x:y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
# with driver.session() as session:
#     for _, row in price_table.iterrows():
#         properties = {kor_cols_map[col]: row[col] for col in eng_colnames}
#         session.run(
#             """
#             MATCH (p:요금제 {고유ID: $product_id})
#             MERGE (price:요금 {월정액: $월정액, 부가세제외월정액: $부가세제외월정액, 
#                                선택약정할인포함부가세제외월정액: $선택약정할인포함부가세제외월정액, 
#                                청구방법: $청구방법, net가격: $net가격})
#             MERGE (p)-[:요금정보]->(price)
#             """,
#             product_id=row['pmProductID'],
#             월정액=properties['월정액'],
#             부가세제외월정액=properties['부가세제외월정액'],
#             선택약정할인포함부가세제외월정액=properties['선택약정할인포함부가세제외월정액'],
#             청구방법=properties['청구방법'],
#             net가격=properties['net가격']
#         )

### VOICE
- 음성 제공량 관련 정보
- Label: 음성통화
- 관계
  - 요금제 -[:제공]-> 음성통화

In [ ]:
voice_table = pd.read_csv(os.path.join(DATA_DIR, "VOICE.csv"))
voice_table.head(1)

In [ ]:
eng_colnames = [
    'includedVoiceCall',
    'includedVideoOrValueAddedCall',
    'includedVoiceCallTospecifiedNumbers',
    'refillAmountRatio',
    'refillRange'
]

kor_colnames = [
    '음성통화제공량',
    '영상및부가통화제공량',
    '지정번호통화제공량',
    '리필비율한도',
    '리필대상'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
# 리스트형으로 변환
voice_table["refillRange"] = voice_table["refillRange"].apply(lambda x: type_cast(x))

# refillAmount 정규화
voice_table["refillAmountRatio"] = voice_table["refillAmount"].apply(lambda x: int(x.replace("%", ""))*0.01 if pd.notna(x) else None)

In [ ]:
voice_table.iloc[109]["refillRange"]

In [ ]:
with driver.session() as session:
    for _, row in voice_table.iterrows():
        # 속성값을 한글 컬럼명으로 변환
        properties = {}
        for col in eng_colnames:
            if type(row[col]) == list:
                properties[kor_cols_map[col]] = row[col]
            elif pd.isna(row[col]):
                # NaN 값을 'null'로 변환 -> 이렇게 해도 되는지 확인 필요 (neo4j에서는 NaN 또는 None을 속성값으로 넣을 수 없음)
                properties[kor_cols_map[col]] = 'null'
            else:
                properties[kor_cols_map[col]] = row[col]
        
        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (v:음성통화 {
                음성통화제공량: $음성통화제공량,
                영상및부가통화제공량: $영상및부가통화제공량,
                지정번호통화제공량: $지정번호통화제공량,
                리필비율한도: $리필비율한도,
                리필대상: $리필대상
            })
            MERGE (p)-[:제공]->(v)
            """,
            product_id=row['pmProductID'],
            음성통화제공량=properties['음성통화제공량'],
            영상및부가통화제공량=properties['영상및부가통화제공량'],
            지정번호통화제공량=properties['지정번호통화제공량'],
            리필비율한도=properties['리필비율한도'],
            리필대상=properties['리필대상']
        )

### SMS
- 문자메시지 제공량 관련 정보
- Label: 문자메시지
- 관계
  - 요금제 -[:제공]-> 문자메시지

In [ ]:
# SMS 데이터 로드
sms_table = pd.read_csv(os.path.join(DATA_DIR, "SMS.csv"))
sms_table.head(1)

In [ ]:
# 컬럼명 매핑 정의
eng_colnames = [
    'includedText',
    'textRange'
]

kor_colnames = [
    '문자제공량',
    '문자대상'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
# 그래프에 저장 -> 'textRange'는 빈 리스트만 존재해서 제외
with driver.session() as session:
    for _, row in sms_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (s:문자메시지 {
                문자제공량: $문자제공량
            })
            MERGE (p)-[:제공]->(s)
            """,
            product_id=row['pmProductID'],
            문자제공량=properties['문자제공량']
        )

### DATA
- 데이터 제공량 관련
- Label: 데이터용량
- 관계
  - 요금제 -[:제공]-> 데이터용량

In [ ]:
data_table = pd.read_csv(os.path.join(DATA_DIR, "DATA.csv"))
data_table.head(1)

In [ ]:
# 컬럼명 매핑 정의
eng_colnames = [
    'includedData', 'includedDataForSharingAndTethering',
    'includedMVoIP', 'appliedSpeed', 'seniorDataExceedAvailable',
    'generalDataExceedAvailable', 'dataRefillAmount',
    'dataRefillCouponGiftingAvailability', 'maximumShareAmount',
    'dataGiftReceivingAvailability'
]

kor_colnames = [
    '기본제공데이터용량', '기본제공데이터중공유및테더링가능용량',
    '기본제공데이터중mvoip용량', '데이터소진후데이터제공속도', '시니어대상데이터소진후최대금액및속도제한적용',
    '데이터소진후최대금액및속도제한적용', '데이터리필가능용량',
    '데이터리필쿠폰선물가능여부', '최대데이터선물가능용량',
    '데이터선물받기가능여부'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
data_table["maximumShareAmount"] = data_table["maximumShareAmount"].apply(lambda x: float(x.replace("GB", "")) if pd.notna(x) else None)

In [ ]:
# data_table에서 NaN을 칼럼의 데이터타입에 맞춰 임의의 값으로 치환
data_table["includedData"] = data_table["includedData"].fillna(0.0)
data_table["includedDataForSharingAndTethering"] = data_table["includedDataForSharingAndTethering"].fillna(0.0)
data_table["includedMVoIP"] = data_table["includedMVoIP"].fillna(0.0)
data_table["appliedSpeed"] = data_table["appliedSpeed"].fillna(0.0)
data_table["seniorDataExceedAvailable"] = data_table["seniorDataExceedAvailable"].fillna("null")
data_table["generalDataExceedAvailable"] = data_table["generalDataExceedAvailable"].fillna("null")
data_table["dataRefillAmount"] = data_table["dataRefillAmount"].fillna(0.0)
data_table["dataRefillCouponGiftingAvailability"] = data_table["dataRefillCouponGiftingAvailability"].fillna("null")
data_table["maximumShareAmount"] = data_table["maximumShareAmount"].fillna(0.0)
data_table["dataGiftReceivingAvailability"] = data_table["dataGiftReceivingAvailability"].fillna("null")

In [ ]:
# 그래프에 저장
with driver.session() as session:
    for _, row in data_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (d:데이터용량 {
            기본제공데이터용량: $기본제공데이터용량,
            기본제공데이터중공유및테더링가능용량: $기본제공데이터중공유및테더링가능용량,
            기본제공데이터중mvoip용량: $기본제공데이터중mvoip용량,
            데이터소진후데이터제공속도: $데이터소진후데이터제공속도,
            시니어대상데이터소진후최대금액및속도제한적용: $시니어대상데이터소진후최대금액및속도제한적용,
            데이터소진후최대금액및속도제한적용: $데이터소진후최대금액및속도제한적용,
            데이터리필가능용량: $데이터리필가능용량,
            데이터리필쿠폰선물가능여부: $데이터리필쿠폰선물가능여부,
            최대데이터선물가능용량: $최대데이터선물가능용량,
            데이터선물받기가능여부: $데이터선물받기가능여부
            })
            MERGE (p)-[:제공]->(d)
            """,
            product_id=row['pmProductID'],
            기본제공데이터용량=properties['기본제공데이터용량'],
            기본제공데이터중공유및테더링가능용량=properties['기본제공데이터중공유및테더링가능용량'],
            기본제공데이터중mvoip용량=properties['기본제공데이터중mvoip용량'],
            데이터소진후데이터제공속도=properties['데이터소진후데이터제공속도'],
            시니어대상데이터소진후최대금액및속도제한적용=properties['시니어대상데이터소진후최대금액및속도제한적용'],
            데이터소진후최대금액및속도제한적용=properties['데이터소진후최대금액및속도제한적용'],
            데이터리필가능용량=properties['데이터리필가능용량'],
            데이터리필쿠폰선물가능여부=properties['데이터리필쿠폰선물가능여부'],
            최대데이터선물가능용량=properties['최대데이터선물가능용량'],
            데이터선물받기가능여부=properties['데이터선물받기가능여부'],
        )

### TOPUP
- 충전 관련
- Label: 충전서비스
- 관계
  - 요금제 -[:가능]-> 충전서비스

In [ ]:
topup_table = pd.read_csv(os.path.join(DATA_DIR, "TOPUP.csv"))
topup_table.head(1)

In [ ]:
# 컬럼명 매핑 정의
eng_colnames = [
    'reChargeAvailability', 'minimumChargeAmount', 'maximumChargeAmount'
]

kor_colnames = [
    '충전서비스대상여부', '최소충전금액', '최대충전금액'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
# NaN 처리
topup_table["minimumChargeAmount"] = topup_table["minimumChargeAmount"].fillna(0.0)
topup_table["maximumChargeAmount"] = topup_table["maximumChargeAmount"].fillna(0.0)

In [ ]:
# 그래프에 저장
with driver.session() as session:
    for _, row in topup_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (t:충전서비스 {
                충전서비스대상여부: $충전서비스대상여부,
                최소충전금액: $최소충전금액,
                최대충전금액: $최대충전금액
            })
            MERGE (p)-[:제공]->(t)
            """,
            product_id=row['pmProductID'],
            충전서비스대상여부=properties['충전서비스대상여부'],
            최소충전금액=properties['최소충전금액'],
            최대충전금액=properties['최대충전금액']
        )

### CUSTOMERCONDITION
- 고객 가입 조건 데이터
- Label: 가입조건
- 관계
  - 요금제 -[:가입조건]-> 가입조건

In [ ]:
condition_table = pd.read_csv(os.path.join(DATA_DIR, "CUSTOMERCONDITION.csv"))
condition_table.iloc[12:16]

In [ ]:
# ageRule 파싱
ageRule_table = []
for _, row in condition_table.iterrows():
    ageRule = ast.literal_eval(row['ageRule'])
    if not ageRule:
        ageRule_table.append(["null", 0, "null", 999])
    else:
        # 초기값 (기본값)
        minAgeCriteria = "null"
        minAge = 0
        maxAgeCriteria = "null"
        maxAge = 999
        for rule in ageRule:
            if "이하" in rule["value"]:
                maxAge = int(regex.findall(r"\d+", rule["value"])[0])
                maxAgeCriteria = regex.findall(r"[일월]기준", rule["value"])[0]
            elif "이상" in rule["value"]:
                minAge = int(regex.findall(r"\d+", rule["value"])[0])
                minAgeCriteria = regex.findall(r"[일월]기준", rule["value"])[0]
        ageRule_table.append([minAgeCriteria, minAge, maxAgeCriteria, maxAge])

In [ ]:
condition_table = pd.concat([condition_table, pd.DataFrame(ageRule_table, columns=["minAgeCriteria", "minAge", "maxAgeCriteria", "maxAge"])], axis=1)

In [ ]:
condition_table.drop(columns=['ageRule'], inplace=True)

In [ ]:
# 리스트형으로 변환
condition_table["customerTypeValueList"] = condition_table["customerTypeValueList"].apply(lambda x: type_cast(x))
condition_table["individualCustomerSubtypeValueList"] = condition_table["individualCustomerSubtypeValueList"].apply(lambda x: type_cast(x))
condition_table["duplicateNameOnboardGroupList"] = condition_table["duplicateNameOnboardGroupList"].apply(lambda x: type_cast(x))

In [ ]:
# NaN 처리
condition_table["customerTypeEligibility"] = condition_table["customerTypeEligibility"].fillna("null")
condition_table["individualCustomerSubtypeEligibility"] = condition_table["individualCustomerSubtypeEligibility"].fillna("null")
condition_table["directPlanOnboard"] = condition_table["directPlanOnboard"].fillna("null")
condition_table["fixedPlanContractConcurrentSignupRestriction"] = condition_table["fixedPlanContractConcurrentSignupRestriction"].fillna("null")
condition_table["tsupportFundOnboard"] = condition_table["tsupportFundOnboard"].fillna("null")
condition_table["duplicateNameOnboardEligibility"] = condition_table["duplicateNameOnboardEligibility"].fillna("null")
condition_table["specialCustomerIsSoldier"] = condition_table["specialCustomerIsSoldier"].fillna("null")
condition_table["specialCustomerIsSoldier"] = condition_table["specialCustomerIsSoldier"].fillna("null")

In [ ]:
condition_table.iloc[12:16]

In [ ]:
eng_colnames = [
    'customerTypeEligibility', 'customerTypeValueList',
    'individualCustomerSubtypeEligibility',
    'individualCustomerSubtypeValueList', 'directPlanOnboard',
    'fixedPlanContractConcurrentSignupRestriction', 'tsupportFundOnboard',
    'duplicateNameOnboardEligibility', 'duplicateNameOnboardGroupList',
    'specialCustomerIsSoldier', 'minAgeCriteria', 'minAge',
    'maxAgeCriteria', 'maxAge'
]

kor_colnames = [
    '고객유형별가입가능여부', '고객유형목록',
    '개인고객세부유형별가입가능여부', 
    '개인고객세부유형목록', '다이렉트플랜가입가능여부',
    '선택약정동시가입가능여부', 'T지원금약정동시가입가능여부',
    '동일명의가입가능여부', '동일명의가입불가그룹목록',
    '군인전용요금제여부', '가입가능최소나이계산기준', '가입가능최소나이',
    '최대나이계산기준', '가입가능최대나이'
]

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
# 그래프에 저장
with driver.session() as session:
    for _, row in condition_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (c:가입조건 {
            고객유형별가입가능여부: $고객유형별가입가능여부,
            고객유형목록: $고객유형목록,
            개인고객세부유형별가입가능여부: $개인고객세부유형별가입가능여부,
            개인고객세부유형목록: $개인고객세부유형목록,
            다이렉트플랜가입가능여부: $다이렉트플랜가입가능여부,
            선택약정동시가입가능여부: $선택약정동시가입가능여부,
            T지원금약정동시가입가능여부: $T지원금약정동시가입가능여부,
            동일명의가입가능여부: $동일명의가입가능여부,
            동일명의가입불가그룹목록: $동일명의가입불가그룹목록,
            군인전용요금제여부: $군인전용요금제여부,
            가입가능최소나이계산기준: $가입가능최소나이계산기준,
            가입가능최소나이: $가입가능최소나이,
            최대나이계산기준: $최대나이계산기준,
            가입가능최대나이: $가입가능최대나이
            })
            MERGE (p)-[:가입조건]->(c)
            """,
            product_id=row['pmProductID'],
            고객유형별가입가능여부=properties['고객유형별가입가능여부'],
            고객유형목록=properties['고객유형목록'],
            개인고객세부유형별가입가능여부=properties['개인고객세부유형별가입가능여부'],
            개인고객세부유형목록=properties['개인고객세부유형목록'],
            다이렉트플랜가입가능여부=properties['다이렉트플랜가입가능여부'],
            선택약정동시가입가능여부=properties['선택약정동시가입가능여부'],
            T지원금약정동시가입가능여부=properties['T지원금약정동시가입가능여부'],
            동일명의가입가능여부=properties['동일명의가입가능여부'],
            동일명의가입불가그룹목록=properties['동일명의가입불가그룹목록'],
            군인전용요금제여부=properties['군인전용요금제여부'],
            가입가능최소나이계산기준=properties['가입가능최소나이계산기준'],
            가입가능최소나이=properties['가입가능최소나이'],
            최대나이계산기준=properties['최대나이계산기준'],
            가입가능최대나이=properties['가입가능최대나이']
        )

### product_group
- 요금제와 그룹간의 관계 정보
- Label: 요금제그룹
- 관계
  - 요금제 -[:속함]-> 요금제그룹

In [ ]:
product_group_table = pd.read_csv(os.path.join(DATA_DIR, "product_group.csv"))
product_group_table.head(1)

In [ ]:
eng_colnames = ['groupName']
kor_colnames = ['그룹명']

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
# 그래프에 저장
with driver.session() as session:
    for _, row in product_group_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        # 그룹 노드 생성 및 요금제와 연결
        session.run(
            """
            MERGE (g:요금제그룹 {그룹명: $그룹명})
            WITH g
            MATCH (p:요금제 {고유ID: $pmProductId})
            MERGE (p)-[:속함]->(g)
            """,
            그룹명=properties['그룹명'],
            pmProductId=row['pmProductId']
        )

### BENEFITCONDITION
- 혜택 조건 정보 -> 구체적인 혜택 정보가 없어서 넣는 의미가 없어보여서 일단 스킵

### DEDUCTIBLE
- 장애인 고객 부가통화 제공량 확대 대상 여부
- Label: 장애인공제
- 관계
  - 요금제 -[:장애인혜택]-> 장애인공제

In [ ]:
deductitble_table = pd.read_csv(os.path.join(DATA_DIR, "DEDUCTIBLE.csv"))
deductitble_table.head(10)

In [ ]:
# Normalize
deductitble_table["additionalOfferForDisabilities"] = deductitble_table["additionalOfferForDisabilities"].apply(lambda x: int(x.replace("분", "")) if pd.notna(x) else 0)

In [ ]:
eng_colnames = ['additionalOfferForDisabilities']
kor_colnames = ['장애인부가통화추가제공량']

kor_cols_map = {x: y for x, y in zip(eng_colnames, kor_colnames)}

In [ ]:
# 그래프에 저장
with driver.session() as session:
    for _, row in deductitble_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}

        session.run(
            """
            MATCH (p:요금제 {고유ID: $product_id})
            MERGE (d:장애인공제 {
                장애인부가통화추가제공량: $장애인부가통화추가제공량
            })
            MERGE (p)-[:장애인혜택]->(d)
            """,
            product_id=row['pmProductID'],
            장애인부가통화추가제공량=properties['장애인부가통화추가제공량']
        )

### 혜택 db
- 혜택정의_6월대상_20250611.xlsx 파일내 '요금제 혜택' 시트 6개를 하나로 통합한 파일 사용 

In [ ]:
benefit_table = pd.read_csv('notebooks/dataload/20250707_benefit_preprocessing.csv')

In [ ]:
benefit_table.isnull().sum()

In [ ]:
benefit_table["appliedDiscountProduct"] = benefit_table["appliedDiscountProduct"].fillna("[]")
benefit_table["discountType"] = benefit_table["discountType"].fillna("")
benefit_table["discountAmount"] = benefit_table["discountAmount"].fillna("")
benefit_table["maxDiscountAmount"] = benefit_table["maxDiscountAmount"].fillna("")

In [ ]:
# 리스트형으로 변환
benefit_table["benefitInformation.marketingKeyword"] = benefit_table["benefitInformation.marketingKeyword"].apply(lambda x: type_cast(x))

In [ ]:
def type_cast_dict(input_data):
    new_data = None
    if pd.isna(input_data):
        input_data = "[]"
    elif '[' in input_data and ']' in input_data and 'nan' in input_data:
        input_data =  input_data.replace("nan", "")
    
    # new_data = [ {"혜택상품명": x["product"], "혜택상품ID": x["pmProductId"]} for x in ast.literal_eval(input_data) ]
    new_data = [ x["product"] for x in ast.literal_eval(input_data) ]

    return new_data

In [ ]:
benefit_table["appliedDiscountProduct"] = benefit_table["appliedDiscountProduct"].apply(lambda x: type_cast_dict(x))

In [ ]:
def str_to_int(input_data):
    if pd.isna(input_data):
        return None
    try:
        return int(input_data.replace("원", ""))
    except ValueError:
        return None

In [ ]:
benefit_table["maxDiscountAmount"] = benefit_table["maxDiscountAmount"].apply(lambda x: str_to_int(x))

In [ ]:
column_map = {
    'benefitInformation.pmBenefitCode': '혜택ID',
    'benefitInformation.benefitName': '혜택명',
    'appliedDiscountProduct':'혜택상품',
    'product': '요금제상품명',
    'pmProductId': '요금제고유ID',
    'legacyProductId': '요금제_SWINGID',
    'discountType': '할인유형',
    'discountAmount': '할인양',
    'maxDiscountAmount': '최대할인금액',
    'benefitType': '혜택유형',
    'benefitInformation.marketingKeyword': '마케팅키워드'
}


In [ ]:
benefit_table = benefit_table.rename(columns=column_map)

In [ ]:
def insert_benefit(tx, row):
    # 아래 필드 제거함
    # b.요금제상품명 = $요금제상품명,
    # b.요금제고유ID = $요금제고유ID,
    # b.요금제_SWINGID = $요금제_SWINGID,
    tx.run("""
    MERGE (b:혜택 {혜택ID: $혜택ID})
    SET
      b.혜택명 = $혜택명,
      b.혜택상품 = $혜택상품,
      b.할인유형 = $할인유형,
      b.할인양 = $할인양,
      b.최대할인금액 = $최대할인금액,
      b.혜택유형 = $혜택유형,
      b.마케팅키워드 = $마케팅키워드

    WITH b
    MATCH (p:요금제 {고유ID: $요금제고유ID})
    MERGE (p)-[:제공혜택]->(b)
    """, **row)

In [ ]:
with driver.session() as session:
    for _, row in benefit_table.iterrows():
        original_row = {k:v for k, v in row.to_dict().items()}
        session.execute_write(insert_benefit, original_row)
        # dict 변환 (NaN 처리)
        # clean_row = {k: (v if pd.notnull(v) else None) for k, v in row.to_dict().items()}
        # session.write_transaction(insert_benefit, clean_row)


### relation_db
- 상품과 혜택(부가서비스)간의 관계
- Label: 부가서비스
- 관계
  - 요금제 -[:??]-> 부가서비스

In [ ]:
relation_db_table = pd.read_csv(os.path.join(DATA_DIR, "relation_db.csv"))
relation_db_table = relation_db_table[(relation_db_table["productId"].notna())&(relation_db_table["productId"] != "-")].copy()
relation_db_table.head()

In [ ]:
relation_type_translation = {
    'productBenefitConditions.allBenefitList': "제공혜택", #부가서비스할인
    'optionData.dataOptionProvidingMethod': "데이터충전", #데이터충전혜택
    'productRelation.signupConcurrentTermination.productList': "가입동시해지",
    'productRelation.signupPreTermination.productList': "가입이전해지",
    'productRelation.terminationConcurrentTermination.productList': "해지동시해지",
    'productRelation.terminationPreTermination.productList': "해지이전해지"
}

In [ ]:
if len(set(relation_db_table["type"].unique()) - relation_type_translation.keys()) != 0:
    raise ValueError("Unknown relation types found in the data.")

In [ ]:
eng_colnames = ["productId", "productName", "type"]
kor_colnames = ["부가서비스ID", "부가서비스명", "관계유형"]
kor_cols_map  = {x: y for x, y in zip(eng_colnames, kor_colnames)}
# 그래프에 저장
with driver.session() as session:
    for _, row in relation_db_table.iterrows():
        properties = {kor_cols_map[col]: row[col] for col in eng_colnames}
        relation_type = relation_type_translation[row['type']]

        if relation_type:
            if '해지' not in relation_type:
                session.run(
                    f"""
                    MATCH (p:요금제 {{고유ID: $product_id}})
                    // MERGE (b:부가서비스 {{부가서비스ID: $부가서비스ID, 부가서비스명: $부가서비스명}})
                    MERGE (b:혜택 {{혜택ID: $부가서비스ID}})
                    ON CREATE SET b.혜택명 = $부가서비스명
                    
                    WITH p, b
                    
                    // MERGE (p)-[:{relation_type}]->(b)
                    WHERE NOT EXISTS((p)-[]->(b))
                    MERGE (p)-[:{relation_type}]->(b)
                    """,
                    product_id=row['pmProductID'],
                    부가서비스ID=properties['부가서비스ID'],
                    부가서비스명=properties['부가서비스명']
                )
            else:
                session.run(
                    f"""
                    MATCH (p:요금제 {{고유ID: $product_id}})
                    MERGE (b:혜택 {{혜택ID: $부가서비스ID}})
                    ON CREATE SET b.혜택명 = $부가서비스명
                    
                    WITH p, b
                    
                    WHERE NOT EXISTS((p)-[]->(b))
                    MERGE (p)-[r:함께가입불가 {{해지시점: '{relation_type}'}}]->(b)
                    """,
                    product_id=row['pmProductID'],
                    부가서비스ID=properties['부가서비스ID'],
                    부가서비스명=properties['부가서비스명']
                )

### 모든 boolean type을 string type으로 변경
- langchain_neo4j의 enhanced_schema를 사용할 때 boolean인 property 하나만 있는 경우 에러나는 버그가 있음

In [ ]:
with driver.session() as session:
    session.run(
        f"""
        MATCH (n)
        UNWIND keys(n) AS key
        WITH n, key, n[key] AS value
        WHERE (value = true OR value = false)
        SET n[key] = CASE WHEN value = true THEN 'true' ELSE 'false' END
        """
    )

In [ ]:
driver.close()

In [ ]:
print("Data loading completed successfully.")

### Benefit 정의
- benefit_table에 있는 혜택들은 BA로 시작하는 '혜택'에 대한 메타데이터임
- pmProductId로 요금제 상품과 연결하고, appliedDiscountProduct 칼럼 내 json 객체의 pmProductId로 부가서비스와 연결함
  - 요금제의 pmProductId는 PA로 시작하고, 혜택상품의 pmProductId는 PB(부가서비스?), PA(요금제 상품), DE(기기), PD(부가서비스?)로 시작함
- 현재는 PA(요금제)->BA(혜택) 관계만 그래프에 저장되어 있음

### relation_db 정의
- relation_db에 있는 상품 관계는 PA로 시작하는 요금제 상품과 BA, MA, NA, PA, PB, PD로 시작하는 것들과의 관계임
  - BA: 혜택 (benefit_table의 혜택과 매핑되는지 확인 필요)
  - MA: ?
  - NA: ?
  - PA: 요금제 상품
  - PB: 부가서비스?
  - PD: 부가서비스?

In [ ]:
relation_db_table[(relation_db_table["productId"].notna())&(relation_db_table["productId"].str.startswith("BA"))]

In [ ]:
relation_db_table[(relation_db_table["productId"].notna())&(relation_db_table["productId"].str.startswith("MA"))]

In [ ]:
benefit_table[["혜택ID", "혜택상품"]].explode("혜택상품").head(10)

In [ ]:
benefit_lines = []
with open('20250707_benefit_preprocessing.csv', 'r') as f:
    for line in f:
        benefit_lines.append(line.strip())

In [ ]:
benefit_table_raw = pd.read_csv('20250707_benefit_preprocessing.csv', dtype=str)

In [ ]:
if "PB0" in benefit_table_raw["appliedDiscountProduct"][0]:
    print(benefit_table_raw["appliedDiscountProduct"][0])

In [ ]:
exist_lines = []
for x in relation_db_table["productId"].unique():
    if pd.isna(x):
        continue
    if x == "-":
        continue
    for idx, row in benefit_table_raw.iterrows():
        for col in benefit_table_raw.columns:
            if pd.notna(row[col]) and x in row[col]:
                exist_lines.append([x, col, row])

In [ ]:
exist_lines_df = pd.DataFrame(exist_lines, columns=["productId", "col", "row"])

In [ ]:
exist_lines_df["col"].unique()

In [ ]:
exist_lines_df